# Joining Spark DataFrames

This notebook demonstrates inner, outer, left, right, left-semi, and left-anti joins with a small product-and-brand dataset.

## Learning objectives

- Identify the left and right sides of a join.
- Predict which matched and unmatched rows each join type returns.
- Explain why semi and anti joins return only columns from the left DataFrame.

## 1. Start Spark before running the notebook

This lesson uses the single-node Spark **standalone cluster**, not `local` mode. In a WSL terminal, start one master and one worker:

```bash
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

Open the master UI at [http://localhost:8080](http://localhost:8080). It should show one live worker. The notebook derives the master URL from the WSL hostname. If your lab uses a different URL, set `SPARK_MASTER` before starting Jupyter.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-DataFrame-Joins")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 2. Create the input DataFrames

`Redmi` has no matching brand. `Sony` has no matching product. These unmatched rows make the join behavior visible.

In [ ]:
products = [
    (1, "iPhone", 100),
    (2, "Galaxy", 200),
    (3, "Redmi", 300),
    (4, "Pixel", 400),
]

brands = [
    (100, "Apple"),
    (200, "Samsung"),
    (400, "Google"),
    (500, "Sony"),
]

product_df = spark.createDataFrame(products, ["product_id", "product_name", "brand_id"])
brand_df = spark.createDataFrame(brands, ["brand_id", "brand_name"])

product_df.show()
brand_df.show()

## 3. Inner join

An inner join keeps only rows whose `brand_id` exists on both sides. Joining by a shared column name also avoids producing two `brand_id` columns.

In [ ]:
product_df.join(brand_df, on="brand_id", how="inner").show()

## 4. Full outer join

A full outer join keeps every row from both sides. Missing values are represented by `NULL` where no match exists.

In [ ]:
product_df.join(brand_df, on="brand_id", how="full").orderBy("brand_id").show()

## 5. Left outer join

The left join keeps every product. A product without a matching brand receives `NULL` for the brand columns.

In [ ]:
product_df.join(brand_df, on="brand_id", how="left").orderBy("product_id").show()

## 6. Right outer join

The right join keeps every brand. A brand without a matching product receives `NULL` for the product columns.

In [ ]:
product_df.join(brand_df, on="brand_id", how="right").orderBy("brand_id").show()

## 7. Left-semi join

A left-semi join answers: *Which left-side rows have at least one match on the right?* It returns only columns from `product_df`. This is similar to SQL `EXISTS`.

In [ ]:
product_df.join(brand_df, on="brand_id", how="left_semi").show()

## 8. Left-anti join

A left-anti join answers: *Which left-side rows have no match on the right?* It also returns only left-side columns. This is useful for finding missing reference data.

In [ ]:
product_df.join(brand_df, on="brand_id", how="left_anti").show()

## 9. Inspect the physical plan

Spark chooses a join strategy based on data size, statistics, and configuration. The exact strategy may differ between environments.

In [ ]:
product_df.join(brand_df, on="brand_id", how="inner").explain(mode="formatted")

## 10. Practice

1. Find brands that have no products.
2. Add a second product for Apple and repeat the semi join. Confirm that a semi join returns each matching left row, not one row per right-side match.

In [ ]:
# Write the practice solution here.

## 11. Stop Spark

Run this cell when the lesson is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")